# Data Preparation (Preparação dos dados)

## Biblioteca / Configuração

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Acesso aos modulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

# Manipulação dos dados
import pandas as pd
import numpy as np
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Diretórios
from configs.paths import *
from configs.function_basic import *

# Pre-processamento
from sklearn.preprocessing import OrdinalEncoder

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print('Ambiente Configurado')

## Parâmetros Globais

In [ ]:
# define a coluna alvo do modelo
TARGET = 'FPD'
# percentual máximo de valores ausentes permitido para manter a variável
PERCENTUAL_MAX_FALTANTES = 20

## Carregamento dos dados 

In [ ]:
# Nome do arquivo
FILE_NAME_RAW = 'book_variaveis_04.parquet'

# Carregar os dados 
abt00 = pd.read_parquet( RAW_DIR / FILE_NAME_RAW)

## Tratamento inicial

### Grupo Controle

In [ ]:
# cria flag para identificar clientes do grupo controle (CPF 6º e 7º dígitos = ZZ ou ZX)
abt00['FLAG_GRUPO_CONTROLE'] = (abt00['NUM_CPF'].astype(str).str[5:7].isin(['ZZ', 'ZX']).astype(int))

controle = abt00[abt00['FLAG_GRUPO_CONTROLE'] == 1]
print(f'Grupo controle: {len(controle):,} registros')

### Definir filtro grupo controle

In [ ]:
# controla aplicação do filtro e remove coluna se só houver grupo controle
APLICAR_FILTRO_PADRAO = True  # True = sem grupo controle | False = base completa

if APLICAR_FILTRO_PADRAO:
    abt01 = abt00[abt00['FLAG_GRUPO_CONTROLE'] == 0].copy()

    # se depois do filtro só existir grupo controle, remove a coluna
    if 'FLAG_GRUPO_CONTROLE' in abt01.columns and abt01['FLAG_GRUPO_CONTROLE'].nunique() == 1:
        abt01.drop(columns=['FLAG_GRUPO_CONTROLE'], inplace=True)
else:
    abt01 = abt00.copy()

print(f"Modo ativo: {'Sem grupo controle' if APLICAR_FILTRO_PADRAO else 'base completa'}")
print(f"Base ativa: {len(abt01):,} registros")


### Separação dos dados para validação temporal (Out-of-Time)

A separação dos dados é realizada com base na **safra**, respeitando a ordem temporal das observações.  
Essa abordagem, conhecida como **validação Out-of-Time (OOT)**, evita vazamento de informação e simula o comportamento real do modelo em dados futuros.

In [ ]:
# garante SAFRA como inteiro
abt01['SAFRA'] = abt01['SAFRA'].astype(int)
safra_counts = abt01['SAFRA'].value_counts().sort_index()

# Definir SAFRAs de teste (Fevereiro e Março 2025)
test_safras = [202502, 202503]

# Criar máscaras
test_mask = abt01['SAFRA'].isin(test_safras)
train_mask = ~test_mask

# Separar dados
train = abt01[train_mask].copy()
test = abt01[test_mask].copy()

train.shape, test.shape

In [ ]:
# Backup dos dados originais
train_01 = train.copy()

# lista de vars para retirar dos tratamentos
ignore_cols = ['SAFRA', 'FPD', 'NUM_CPF', 'DATADENASCIMENTO', 'DATA_SAFRA']

# Aplicando no treino
train_01 = train_01.drop(columns=ignore_cols)

In [ ]:
print('metadados'.upper())
print('=' * 30)

metadados = dataset_info_table(train_01)

### Remoção de Colunas Desnecessárias

In [ ]:
# filtra variáveis com muitos nulos OU cardinalidade igual a 1
df_low_card = metadados[(metadados['PC_nulos'] >= PERCENTUAL_MAX_FALTANTES) | (metadados['Cardinalidade'] <= 1)]
df_low_card = list(df_low_card.Feature.values)

# efeito real do drop
qtd_excluir = train_01.columns.isin(df_low_card).sum()

print(qtd_excluir)
print(df_low_card)

In [ ]:
# filtra variáveis com muitos nulos OU cardinalidade igual a 1
df_high_card = metadados[(metadados['Cardinalidade'] >= 1000)]
df_high_card = list(df_high_card.Feature.values)

# efeito real do drop
qtd_excluir = train_01.columns.isin(df_high_card).sum()

print(qtd_excluir)
print(df_high_card)

In [ ]:
# Unir colunas de baixa e alta cardinalidade
drop_card = list(set(df_low_card + df_high_card))
# Remover colunas de baixa e alta cardinalidade
train_01 = train_01.drop(columns=drop_card, errors='ignore')

In [ ]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'drop_card.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(drop_card, f)

### Cardinalidade (Variáveis)

In [ ]:
# configurar encoder ordinal robusto a categorias novas
oe = OrdinalEncoder( handle_unknown='use_encoded_value', unknown_value=-1)

# treinar encoder e transformar coluna de texto em inteiro
train_01['REGIAO_POSTAL_TXT_enc'] = oe.fit_transform( train_01[['REGIAO_POSTAL_TXT']]).astype(int)

In [ ]:
# Remover coluna
train_01.drop(columns=['REGIAO_POSTAL_TXT'], inplace=True)

In [ ]:
# salvar encoder treinado em arquivo pkl
artifact_path = Path(ARTIFACT_DIR) / "ordinal_encoder_regiao_postal.pkl"

with open(artifact_path, 'wb') as f:
    pickle.dump(oe, f)

In [ ]:
# objetivo: extrair flags sem explodir cardinalidade
train_01['VAR25_FUNC_PRIV'] = train_01['var_25'].str.contains('FUNC_PRIVADO', na=False).astype(int)
train_01['VAR25_FUNC_PUBL'] = train_01['var_25'].str.contains('FUNC_PUBL', na=False).astype(int)
train_01['VAR25_APOSENTADO'] = train_01['var_25'].str.contains('APOSENTADO', na=False).astype(int)
train_01['VAR25_AUX_EMRG'] = train_01['var_25'].str.contains('AUX_EMRG', na=False).astype(int)
train_01['VAR25_BOLSA_FAM'] = train_01['var_25'].str.contains('BOLSA_FAMILIA', na=False).astype(int)
train_01['VAR25_EMPREENDEDOR'] = train_01['var_25'].str.contains('EMPR/DIRETOR', na=False).astype(int)

# Remover coluna
train_01.drop(columns=['var_25'], inplace=True)

### Tratamento de Valores Faltantes

In [ ]:
# Ajustes para gerar o metadados
# Identificar colunas que possuem o valor 'Desconhecido' em pelo menos uma linha
cols_com_desconhecido = [c for c in train_01.columns if train_01[c].astype(str).eq('Desconhecido').any()]

# Tratar valores desconhecidos como missing
for col in cols_com_desconhecido:
    train_01[col] = (train_01[col].astype(str).str.strip().replace('Desconhecido', np.nan))

In [ ]:
# converter todas as colunas para float64
train_01 = train_01.astype('float64')

In [ ]:
# Análise de missing values restantes
train_01, stats = custom_fillna(train_01, strategy='median')

In [ ]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'stats_nulo.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(stats, f)

## Preparando os dados de Teste

In [ ]:
# Backup dos dados originais
test_01 = test.copy()

# lista de vars para retirar dos tratamentos
ignore_cols = ['SAFRA', 'FPD', 'NUM_CPF', 'DATADENASCIMENTO', 'DATA_SAFRA']

# Aplicando no treino
test_01 = test_01.drop(columns=ignore_cols)

### Remoção de Colunas Desnecessárias

In [ ]:
# Aplicar lista top features no teste
with open(Path(ARTIFACT_DIR) /'drop_card.pkl', 'rb') as f:
    drop_card= pickle.load(f)

test_01 = test_01.drop(columns=drop_card, errors='ignore')

### Cardinalidade (Variáveis)

In [ ]:
# carregar encoder salvo para reaplicação consistente
with open(Path(ARTIFACT_DIR) /'ordinal_encoder_regiao_postal.pkl', 'rb') as f:
    cols_drop_fs = pickle.load(f)


# aplicar encoder carregado no dataset de teste
test_01['REGIAO_POSTAL_TXT_enc'] = oe.transform(test_01[['REGIAO_POSTAL_TXT']]).astype(int)


In [ ]:
# Remover coluna
test_01.drop(columns=['REGIAO_POSTAL_TXT'], inplace=True)

In [ ]:
# objetivo: extrair flags sem explodir cardinalidade
test_01['VAR25_FUNC_PRIV'] = test_01['var_25'].str.contains('FUNC_PRIVADO', na=False).astype(int)
test_01['VAR25_FUNC_PUBL'] = test_01['var_25'].str.contains('FUNC_PUBL', na=False).astype(int)
test_01['VAR25_APOSENTADO'] = test_01['var_25'].str.contains('APOSENTADO', na=False).astype(int)
test_01['VAR25_AUX_EMRG'] = test_01['var_25'].str.contains('AUX_EMRG', na=False).astype(int)
test_01['VAR25_BOLSA_FAM'] = test_01['var_25'].str.contains('BOLSA_FAMILIA', na=False).astype(int)
test_01['VAR25_EMPREENDEDOR'] = test_01['var_25'].str.contains('EMPR/DIRETOR', na=False).astype(int)

# Remover coluna
test_01.drop(columns=['var_25'], inplace=True)

### Tratamento de Valores Faltantes

In [ ]:
# Ajustes para gerar o metadados
# Identificar colunas que possuem o valor 'Desconhecido' em pelo menos uma linha
cols_com_desconhecido = [c for c in test_01.columns if test_01[c].astype(str).eq('Desconhecido').any()]

# Tratar valores desconhecidos como missing
for col in cols_com_desconhecido:
    test_01[col] = (test_01[col].astype(str).str.strip().replace('Desconhecido', np.nan))

In [ ]:
# converter todas as colunas para float64
test_01 = test_01.astype('float64')

In [ ]:
# Carregar stats do treino
with open(Path(ARTIFACT_DIR) / 'stats_nulo.pkl', 'rb') as f:
    stats = pickle.load(f)

# Preencher colunas numéricas
for col, value in stats['numerical'].items():
    if col in test_01.columns:
        test_01[col] = test_01[col].fillna(value)

# Preencher colunas categóricas
for col in stats['categorical_cols']:
    if col in test_01.columns:
        test_01[col] = test_01[col].fillna(stats['categorical_fill'])

## Validação

In [ ]:
# Colunas do treino e teste
cols_train = set(train_01.columns)
cols_test = set(test_01.columns)

# Verifica se há diferença
print("Colunas diferentes entre treino e teste:", cols_train.symmetric_difference(cols_test))

In [ ]:
# Comparar tipos das colunas
for col in train_01.columns:
    if col in test_01.columns:
        if train_01[col].dtype != test_01[col].dtype:
            print(f"Diferente dtype: {col} -> treino: {train_01[col].dtype}, teste: {test_01[col].dtype}")

### Checar valores nulos

In [ ]:
# Checar se o teste ainda tem nulos em colunas importantes
print(test_01.isna().sum()[test_01.isna().sum() > 0])

In [ ]:
# Alinha a ordem das colunas do teste com o treino para garantir consistência no modelo
test_01 = test_01[train_01.columns]

In [ ]:
# Reanexar target e o controle temporal antes de salvar

# Para treino
abt01_train_final = train_01.copy()
abt01_train_final['SAFRA'] = train['SAFRA']
abt01_train_final[TARGET] = train[TARGET]


# Para teste
abt01_test_final = test_01.copy()
abt01_test_final['SAFRA'] = test['SAFRA']
abt01_test_final[TARGET] = test[TARGET]


## Salvamento dos Dados Processados

In [ ]:
# Salvar datasets processados em CSV
print('\n💾 Salvando dados processados...')

abt01_train_final.to_csv(PROCESSED_DIR / 'abt01_train.csv', index=False)
abt01_test_final.to_csv(PROCESSED_DIR / 'abt01_test.csv', index=False)

print(f'   ✓ Treino salvo: {PROCESSED_DIR / "abt01_train.csv"}')
print(f'   ✓ Teste salvo: {PROCESSED_DIR / "abt01_test.csv"}')

# Salvar lista de features (excluindo a target) para referência futura
selected_features_abt01 = [c for c in abt01_train_final.columns if c != TARGET]
with open(ARTIFACT_DIR / 'selected_features_abt01.pkl', 'wb') as f:
    pickle.dump(selected_features_abt01, f)

print(f'   ✓ Lista de features salva em: {ARTIFACT_DIR / "selected_features_abt01.pkl"}')
print(f'\n✅ Todos os dados processados foram salvos')
